In [ ]:
# Install astronomy helpers (silent)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sunpy[all]', 'astropy', 'pyerfa'],
               capture_output=True)
print("✓ Dependencies ready")

In [ ]:
import os, warnings, glob, time
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output, display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from astropy.io import fits
from astropy.time import Time

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

SEED = 2024
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f"✓ Device  : {device}")
print(f"✓ GPUs    : {NUM_GPUS}")
if device.type == 'cuda':
    for i in range(NUM_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}  : {p.name}  ({p.total_memory/1e9:.1f} GB)")
print(f"✓ PyTorch : {torch.__version__}")

In [ ]:
C_THRESHOLD = 35.0
M_THRESHOLD = 131.4
X_THRESHOLD = 1_000.0

CADENCE_S      = 10
HISTORY_STEPS  = 360        # 60-min input (Mamba handles long context cheaply)
FORECAST_STEPS = 60         # 10-min prediction horizon

TCN_CHANNELS   = [64, 128, 128]
TCN_KERNEL     = 3
TCN_DILATIONS  = [1, 2, 4, 8, 16]
TCN_DROPOUT    = 0.10

MAMBA_D_MODEL  = 128
MAMBA_D_STATE  = 16
MAMBA_D_CONV   = 4
MAMBA_EXPAND   = 2

BATCH_SIZE   = 256
EPOCHS       = 30
LR           = 2e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP    = 1.0

W_REG    = 0.80
W_CLASS  = 0.60
W_PEAK   = 1.50
W_NEUPERT = 0.10

FOCAL_GAMMA  = 2.0
WING_W       = 10.0
WING_EPS     = 2.0

W_DELTA      = 0.40   # weight for first-difference (shape) loss
W_CURV       = 0.15   # weight for second-difference (curvature) loss

W_PEAK_FOCAL = 2.0    # additional loss multiplier for non-quiet forecast windows

EMA_ALPHA    = 0.15   # higher = more smoothing; 0.15 ≈ ~2-min half-life at 10s cadence
                      # Set to 0.0 to disable EMA smoothing entirely

SOLEXS_DIR = 'PATH_TO_SOLEXS_DATA'
HELIOS_DIR = 'PATH_TO_HELIOS_DATA'
GTI_DIR    = 'PATH_TO_GTI'

FEATURE_COLS_V4 = []   
INPUT_DIM       = None  

print("✓ Configuration v4 loaded")
print(f"  History  : {HISTORY_STEPS * CADENCE_S // 60} min  ({HISTORY_STEPS} steps)")
print(f"  Forecast : {FORECAST_STEPS * CADENCE_S // 60} min ({FORECAST_STEPS} steps)")
print(f"  GTI dir  : {GTI_DIR}")

In [ ]:
_ADITYA_EPOCH = pd.Timestamp('2024-01-01T00:00:00')

def load_fits_table(path):
    # Return the first binary-table extension of a FITS file as a DataFrame
    with fits.open(path) as hdul:
        for ext in hdul:
            if not ext.is_image and ext.data is not None:
                df = pd.DataFrame(ext.data)
                df.columns = [c.strip().upper() for c in df.columns]
                return df
    raise ValueError(f"No table extension in {path}")

def parse_solexs_file(path):
    df = load_fits_table(path)
    tcol = 'TIME' if 'TIME' in df.columns else df.columns[0]
    rcol = next((c for c in ('COUNTS', 'RATE', 'CTR') if c in df.columns), df.columns[1])
    out = pd.DataFrame({
        'time'        : pd.to_datetime(df[tcol], unit='s').dt.tz_localize(None),
        'solexs_rate' : df[rcol].astype(float).clip(lower=0)
    })
    return out.sort_values('time').reset_index(drop=True)


# Priority lookup: FITS column name → canonical detector name
_HX_COL_MAP = {
    'CTR_CDTE1':   'hx_cdte1', 'RATE_CDTE1':   'hx_cdte1',
    'COUNT_CDTE1': 'hx_cdte1', 'CDTE1':         'hx_cdte1',
    'CTR_CDTE2':   'hx_cdte2', 'RATE_CDTE2':    'hx_cdte2',
    'COUNT_CDTE2': 'hx_cdte2', 'CDTE2':         'hx_cdte2',
    'CTR_CZT1':    'hx_czt1',  'RATE_CZT1':     'hx_czt1',
    'COUNT_CZT1':  'hx_czt1',  'CZT1':          'hx_czt1',
}
# Filename substrings → canonical detector (for per-file detector layout)
_HX_FNAME_MAP = [
    ('cdte1', 'hx_cdte1'), ('cdte2', 'hx_cdte2'), ('czt1', 'hx_czt1'),
    ('cdte',  'hx_cdte1'), ('czt',   'hx_czt1'),
]
_HX_RATE_KW  = ('CTR', 'COUNTS', 'RATE', 'COUNT_RATE')
_HX_TIME_KW  = {'TIME', 'ISOT', 'MJD', 'TIMEZERO'}


def parse_helios_file(path):
    df    = load_fits_table(path)
    fname = os.path.basename(path).lower()

    if 'ISOT' in df.columns:
        times = pd.to_datetime(df['ISOT']).dt.tz_localize(None)
    elif 'MJD' in df.columns:
        times = pd.to_datetime(Time(df['MJD'].values, format='mjd').datetime).dt.tz_localize(None)
    elif 'TIME' in df.columns:
        # times = pd.to_datetime(df['TIME'], unit='s').dt.tz_localize(None)
        times = pd.to_timedelta(df['TIME'], unit='s')
    else:
        times = pd.to_datetime(df[df.columns[0]]).dt.tz_localize(None)

    out = pd.DataFrame({'time': times})

    mapped = {
        _HX_COL_MAP[c]: df[c].astype(float).clip(lower=0)
        for c in df.columns if c in _HX_COL_MAP
    }
    if mapped:
        for det_name, series in sorted(mapped.items()):
            out[det_name] = series.values
        return out.sort_values('time').reset_index(drop=True)

    det_name = next(
        (canon for token, canon in _HX_FNAME_MAP if token in fname),
        None
    )
    rate_col = next(
        (c for c in df.columns
         if any(kw in c for kw in _HX_RATE_KW) and c not in _HX_TIME_KW),
        None
    )
    if rate_col is None:
        non_time = [c for c in df.columns if c not in _HX_TIME_KW]
        rate_col = non_time[0] if non_time else df.columns[-1]
    if det_name is None:
        det_name = 'hx_cdte1'
        print(f"  [HEL1OS] Cannot infer detector from {os.path.basename(path)!r}; "
              f"assigning to '{det_name}'")
    out[det_name] = df[rate_col].astype(float).clip(lower=0).values
    return out.sort_values('time').reset_index(drop=True)


def load_payload_data(solexs_dir, helios_dir, cadence='10S'):
    sx_files = glob.glob(os.path.join(solexs_dir, '*'))
    hx_files = glob.glob(os.path.join(helios_dir, '*'))
    print(f"  SoLEXS files : {len(sx_files)}")
    print(f"  HEL1OS files : {len(hx_files)}")

    sx_dfs, hx_dfs = [], []
    for f in sx_files:
        try:   sx_dfs.append(parse_solexs_file(f))
        except Exception as e: print(f"  [skip SoLEXS] {os.path.basename(f)}: {e}")
    for f in hx_files:
        try:   hx_dfs.append(parse_helios_file(f))
        except Exception as e: print(f"  [skip HEL1OS] {os.path.basename(f)}: {e}")

    if not sx_dfs: raise FileNotFoundError("No SoLEXS files parsed.")
    if not hx_dfs: raise FileNotFoundError("No HEL1OS files parsed.")

    df_sx = (pd.concat(sx_dfs).drop_duplicates('time')
               .set_index('time').resample(cadence).mean()
               .interpolate('linear'))

    # Concat all HEL1OS frames; average exact duplicate timestamps per detector;
    # then resample — each detector column is preserved independently.
    hx_all = pd.concat(hx_dfs).set_index('time')
    df_hx  = (hx_all.groupby(level=0).mean()
                     .resample(cadence).mean()
                     .interpolate('linear'))

    hx_det_cols = sorted(c for c in df_hx.columns if c.startswith('hx_'))
    print(f"  HEL1OS detectors: {hx_det_cols}")

    df = (df_sx.join(df_hx, how='outer')
               .interpolate('linear').fillna(0).clip(lower=0))
    print(f"  Merged span : {df.index[0]} → {df.index[-1]}  ({len(df):,} rows)")
    return df

print("✓ FITS loaders defined  (multi-detector HEL1OS)")

In [ ]:
def parse_gti_file(path):
    with fits.open(path) as hdul:
        for ext in hdul:
            if not ext.is_image and ext.data is not None:
                col_names = [c.strip().upper() for c in ext.columns.names]
                data      = ext.data

                start_col = next(
                    (c for c in ('START', 'TSTART', 'START_TIME') if c in col_names),
                    col_names[0]
                )
                stop_col  = next(
                    (c for c in ('STOP', 'TSTOP', 'STOP_TIME') if c in col_names),
                    col_names[1]
                )
                return pd.DataFrame({
                    'start': data[start_col].astype(np.float64),
                    'stop':  data[stop_col].astype(np.float64),
                })
    raise ValueError(f"No binary-table extension found in GTI file: {path}")


# Aditya-L1 Mission Elapsed Time reference epoch
_ADITYA_EPOCH = pd.Timestamp('2024-01-01T00:00:00')


def build_gti_mask(gti_dir, time_index, cadence_s=CADENCE_S):
    # Search for GTI files with common naming conventions
    search_patterns = ['*.gti', '*.GTI', '*_gti.fits', '*_GTI.fits',
                       '*gti*.fits', '*GTI*.fits']
    gti_files = []
    for pat in search_patterns:
        gti_files.extend(glob.glob(os.path.join(gti_dir, pat)))
    gti_files = sorted(set(gti_files))
    gti_files = [f for f in gti_files if 'sdd1' not in os.path.basename(f).lower()]
    if not gti_files:
        print(f"  ⚠  No GTI files found in {gti_dir!r} — mask=1.0 (all valid)")
        return pd.Series(np.ones(len(time_index), dtype=np.float32), index=time_index)

    all_intervals = []
    for path in gti_files:
        try:
            df_g = parse_gti_file(path)
            all_intervals.append(df_g)
            print(f"  GTI  {os.path.basename(path)}: {len(df_g)} intervals "
                  f"({df_g['start'].min():.1f}s – {df_g['stop'].max():.1f}s MET)")
        except Exception as e:
            print(f"  [skip GTI] {os.path.basename(path)}: {e}")

    if not all_intervals:
        print("  ⚠  All GTI files failed to parse — mask=1.0")
        return pd.Series(np.ones(len(time_index), dtype=np.float32), index=time_index)

    gti_all = pd.concat(all_intervals, ignore_index=True).sort_values('start')

    # Convert MET → nanosecond integers for fast vectorised comparison
    t_start_ns = (
         pd.to_timedelta(gti_all['start'].values, unit='s')
    ).view(np.int64)
    t_stop_ns  = (
         pd.to_timedelta(gti_all['stop'].values, unit='s')
    ).view(np.int64)
    t_idx_ns = time_index.view(np.int64)   # nanoseconds since Unix epoch

    mask = np.zeros(len(time_index), dtype=np.float32)
    for t0, t1 in zip(t_start_ns, t_stop_ns):
        mask[(t_idx_ns >= t0) & (t_idx_ns <= t1)] = 1.0

    n_valid  = int(mask.sum())
    pct      = 100.0 * n_valid / len(mask)
    n_files  = len(gti_files)
    n_parsed = len(all_intervals)
    print(f"  GTI mask : {n_valid:,}/{len(mask):,} valid steps ({pct:.1f}%)  "
          f"[{n_parsed}/{n_files} files parsed, {len(gti_all)} total intervals]")
    return pd.Series(mask, index=time_index, dtype=np.float32)


print("✓ GTI parser defined  (parse_gti_file + build_gti_mask)")
print(f"  Reference epoch: {_ADITYA_EPOCH}  (Aditya-L1 MET origin)")

In [ ]:
def _inject_flare(signal, n, t0, amplitude, rise_steps, decay_steps):
    """Solar flare profile: quadratic rise + exponential decay."""
    peak = t0 + rise_steps
    for t in range(max(0, t0), min(n, peak)):
        signal[t] += amplitude * ((t - t0) / max(rise_steps, 1)) ** 2
    tau = max(decay_steps / 2.5, 1)
    for t in range(min(n, peak), min(n, peak + decay_steps * 2)):
        signal[t] += amplitude * np.exp(-(t - peak) / tau)


def create_synthetic_solar_data(n_days=60, cadence_s=10, seed=SEED):
    rng = np.random.default_rng(seed)
    n   = int(n_days * 86_400 / cadence_s)
    t   = np.arange(n) * cadence_s

    bg = ( 6.0
           + 2.0 * np.sin(2 * np.pi * t / 660)
           + 1.5 * np.sin(2 * np.pi * t / 3600)
           + rng.normal(0, 0.4, n) )
    bg = np.clip(bg, 0.5, None)

    sx_flare = np.zeros(n)
    hx_flare = np.zeros(n)   # shared non-thermal impulse base

    def add_flares(n_events, amp_lo, amp_hi, rise_lo, rise_hi, decay_lo, decay_hi):
        for _ in range(n_events):
            t0    = rng.integers(200, n - 3_000)
            amp   = rng.uniform(amp_lo, amp_hi)
            rise  = int(rng.uniform(rise_lo, rise_hi)   * 60 / cadence_s)
            decay = int(rng.uniform(decay_lo, decay_hi) * 60 / cadence_s)
            _inject_flare(sx_flare, n, t0, amp, rise, decay)
            hx_amp = amp * rng.uniform(0.25, 0.55)
            _inject_flare(hx_flare, n, t0 + rise // 2, hx_amp,
                          max(1, rise // 3), decay // 2)

    add_flares(n_days * 8,        25,   100,  2,  8,  8,  35)   # C-class
    add_flares(int(n_days * 1.5), 100, 1_100, 4, 15, 15,  70)   # M-class
    add_flares(max(2, n_days//5), 1_100,7_000, 5, 20, 30, 120)  # X-class

    solexs_rate = np.clip(bg + sx_flare, 0.1, None)

    hx_cdte1 = np.clip(rng.normal(0, 0.80, n) + hx_flare, 0, None)

    gain2    = rng.uniform(0.88, 0.97)
    hx_cdte2 = np.clip(rng.normal(0, 1.10, n) + hx_flare * gain2, 0, None)

    # ── CZT-1: softer hard X-ray (5-30 keV proxy) — delayed 3 cadence steps ──
    # CZT sees more thermal-blend and responds slightly later than CdTe
    hx_flare_delayed = np.roll(hx_flare, 3)       # 30-s causal delay
    hx_czt1 = np.clip(rng.normal(0, 0.50, n) + hx_flare_delayed * 0.60, 0, None)

    dates = pd.date_range('2024-01-01', periods=n, freq=f'{cadence_s}S')
    df = pd.DataFrame({
        'solexs_rate': solexs_rate,
        'hx_cdte1'   : hx_cdte1,
        'hx_cdte2'   : hx_cdte2,
        'hx_czt1'    : hx_czt1,
    }, index=dates)

    print(f"✓ Synthetic data: {n_days} days  |  {n:,} steps  |  cadence {cadence_s}s")
    tot = solexs_rate
    c_ev = int(np.sum(tot > C_THRESHOLD) * cadence_s / 300)
    m_ev = int(np.sum(tot > M_THRESHOLD) * cadence_s / 600)
    x_ev = int(np.sum(tot > X_THRESHOLD) * cadence_s / 1200)
    print(f"  ≥C: ~{c_ev} events  |  ≥M: ~{m_ev}  |  ≥X: ~{x_ev}")
    print(f"  SoLEXS  min={tot.min():.1f}  mean={tot.mean():.1f}  max={tot.max():.1f}")
    print(f"  HEL1OS channels: hx_cdte1  hx_cdte2  hx_czt1")
    return df

print("✓ Synthetic data generator defined  (SoLEXS + 3 HEL1OS detectors)")

In [ ]:
def build_features_v4(df, gti_mask_series=None):
    d = df.copy()

    # Discover HEL1OS detector columns (canonical names start with 'hx_')
    hx_det_cols = sorted(c for c in df.columns if c.startswith('hx_'))
    if not hx_det_cols:
        raise ValueError("No HEL1OS detector columns found. "
                         "Expected columns with 'hx_' prefix (e.g. hx_cdte1).")

    feat_cols = []

    # 1. SoLEXS log1p  (compresses 4+ orders of magnitude)
    sx_log_raw = np.log1p(d['solexs_rate'].values.clip(0))
    # ── EMA smoothing on sx_log (suppresses shot noise, preserves flare onset) ──
    # alpha=EMA_ALPHA ≈ 2-min half-life at 10s cadence; set EMA_ALPHA=0 to disable.
    if EMA_ALPHA > 0:
        ema = np.empty_like(sx_log_raw)
        ema[0] = sx_log_raw[0]
        a = EMA_ALPHA
        for i in range(1, len(sx_log_raw)):
            ema[i] = a * sx_log_raw[i] + (1.0 - a) * ema[i - 1]
        d['sx_log'] = ema
    else:
        d['sx_log'] = sx_log_raw
    feat_cols.append('sx_log')

    # 2. Per-detector HEL1OS log1p (with same EMA smoothing)
    for det in hx_det_cols:
        suf = det[3:]           # strip 'hx_' → 'cdte1', 'cdte2', 'czt1'
        col = f'hx_log_{suf}'
        hx_raw = np.log1p(d[det].values.clip(0))
        if EMA_ALPHA > 0:
            ema_hx = np.empty_like(hx_raw)
            ema_hx[0] = hx_raw[0]
            for i in range(1, len(hx_raw)):
                ema_hx[i] = a * hx_raw[i] + (1.0 - a) * ema_hx[i - 1]
            d[col] = ema_hx
        else:
            d[col] = hx_raw
        feat_cols.append(col)

    # 3. SoLEXS first derivative  (thermal emission rate)
    d['sx_delta'] = d['sx_log'].diff().fillna(0)
    feat_cols.append('sx_delta')

    # 4. Per-detector HEL1OS first derivative  (non-thermal impulse — Neupert proxy)
    for det in hx_det_cols:
        suf = det[3:]
        col = f'hx_delta_{suf}'
        d[col] = d[f'hx_log_{suf}'].diff().fillna(0)
        feat_cols.append(col)

    # 5. SoLEXS second derivative  (acceleration of thermal emission)
    d['sx_accel'] = d['sx_delta'].diff().fillna(0)
    feat_cols.append('sx_accel')

    # 6. Per-detector impulsive-to-thermal ratio (hard / soft)
    for det in hx_det_cols:
        suf = det[3:]
        col = f'hx_sx_ratio_{suf}'
        d[col] = (d[f'hx_log_{suf}'] / (d['sx_log'] + 1e-3)).clip(0, 5)
        feat_cols.append(col)

    # 7. GTI mask — MUST be last (FeatureNormalizer skips normalising index [-1])
    if gti_mask_series is not None:
        d['gti_mask'] = gti_mask_series.reindex(d.index, fill_value=0).values.astype(np.float32)
    else:
        d['gti_mask'] = 1.0   # synthetic / GTI-unavailable → all valid
    feat_cols.append('gti_mask')

    # Regression target = log1p(SoLEXS), same space as sx_log
    d['target'] = d['sx_log']

    return d.fillna(0), feat_cols


class FeatureNormalizer:
    def __init__(self):
        self.mean_ = None
        self.std_  = None

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0) + 1e-8
        self.mean_[-1] = 0.0   # GTI mask: pass through unchanged
        self.std_[-1]  = 1.0

    def transform(self, X):
        return (X - self.mean_) / self.std_

    def inverse_target(self, y_norm):
        """sx_log occupies channel [0] regardless of N detectors."""
        y_log1p = y_norm * self.std_[0] + self.mean_[0]
        return np.expm1(np.clip(y_log1p, 0, None))

    def log1p_thresholds(self):
        raw = np.array([np.log1p(C_THRESHOLD),
                        np.log1p(M_THRESHOLD),
                        np.log1p(X_THRESHOLD)])
        return (raw - self.mean_[0]) / self.std_[0]


print("✓ Preprocessing v4 defined  (dynamic multi-detector features)")
print("  build_features_v4() returns (df_out, feat_cols) — INPUT_DIM set in pipeline")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class SolarFlareDataset(Dataset):

    def __init__(self, X_norm, y_norm, history=HISTORY_STEPS,
                 forecast=FORECAST_STEPS, thresholds_norm=None):
        self.X    = torch.tensor(X_norm, dtype=torch.float32)
        self.y    = torch.tensor(y_norm, dtype=torch.float32)
        self.H    = history
        self.F    = forecast
        self.N    = len(X_norm) - history - forecast + 1
        self.thrs = thresholds_norm
        # gti_mask is always the last feature channel — inferred from tensor width
        self.gti_col = X_norm.shape[1] - 1

        weights = np.ones(self.N,  dtype=np.float32)
        labels  = np.zeros(self.N, dtype=np.int64)
        # peaked[i] = True when the forecast window has a clear rise-then-fall
        # (peak occurs before the last 30% of forecast steps → turning point present)
        self.peaked = np.zeros(self.N, dtype=bool)
        if self.thrs is not None:
            t_C, t_M, t_X = self.thrs
            for i in range(self.N):
                fut = y_norm[i + history : i + history + forecast]
                peak = fut.max()
                peak_idx = fut.argmax()
                # Mark as peaked if peak occurs in first 70% of forecast window
                if peak_idx < int(forecast * 0.70) and peak_idx > 0:
                    self.peaked[i] = True
                if   peak >= t_X: weights[i], labels[i] = 60.0, 3
                elif peak >= t_M: weights[i], labels[i] = 20.0, 2
                elif peak >= t_C: weights[i], labels[i] =  4.0, 1
                # Extra boost for peaked windows (curve-shape learning)
                if self.peaked[i]:
                    weights[i] *= 3.0
        self.weights = torch.tensor(weights, dtype=torch.float32)
        self.labels  = torch.tensor(labels,  dtype=torch.long)

    def __len__(self): return self.N

    def __getitem__(self, idx):
        x        = self.X[idx : idx + self.H]
        y        = self.y[idx + self.H : idx + self.H + self.F]
        gti_mask = x[:, self.gti_col]          # [H]  — 0/1
        return x, y, gti_mask, self.weights[idx], self.labels[idx]


def compute_focal_weights(dataset, sqrt_scale=True, x_boost=1.5):
    labels = dataset.labels.numpy()
    raw = compute_class_weight('balanced', classes=np.arange(4), y=labels)
    w = np.sqrt(raw) if sqrt_scale else raw
    w[3] *= x_boost   # boost X-class (very rare)
    return torch.tensor(w, dtype=torch.float32)

def make_loaders_v4(df_feat, normalizer, feature_cols=None,
                    val_frac=0.15, test_frac=0.05):
    """
    Chronological split → DataLoaders with pin_memory + num_workers=4.
    feature_cols : explicit list (from build_features_v4); defaults to FEATURE_COLS_V4.
    """
    feat_cols = feature_cols if feature_cols is not None else FEATURE_COLS_V4
    X_raw = df_feat[feat_cols].values.astype(np.float32)
    y_raw = df_feat['target'].values.astype(np.float32)

    N       = len(X_raw)
    n_test  = max(HISTORY_STEPS + FORECAST_STEPS + 1, int(N * test_frac))
    n_val   = max(HISTORY_STEPS + FORECAST_STEPS + 1, int(N * val_frac))
    n_train = N - n_val - n_test

    X_tr, y_tr = X_raw[:n_train],               y_raw[:n_train]
    X_vl, y_vl = X_raw[n_train:n_train+n_val],  y_raw[n_train:n_train+n_val]
    X_te, y_te = X_raw[n_train+n_val:],          y_raw[n_train+n_val:]

    normalizer.fit(X_tr)
    X_tr = normalizer.transform(X_tr)
    X_vl = normalizer.transform(X_vl)
    X_te = normalizer.transform(X_te)

    y_tr_n = (y_tr - normalizer.mean_[0]) / normalizer.std_[0]
    y_vl_n = (y_vl - normalizer.mean_[0]) / normalizer.std_[0]
    y_te_n = (y_te - normalizer.mean_[0]) / normalizer.std_[0]

    thrs   = normalizer.log1p_thresholds()
    ds_tr  = SolarFlareDataset(X_tr, y_tr_n, thresholds_norm=thrs)
    ds_vl  = SolarFlareDataset(X_vl, y_vl_n, thresholds_norm=thrs)
    ds_te  = SolarFlareDataset(X_te, y_te_n, thresholds_norm=thrs)

    sampler = torch.utils.data.WeightedRandomSampler(
        ds_tr.weights, num_samples=min(len(ds_tr), 300_000), replacement=True)

    tr_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler, persistent_workers=True,
                           num_workers=2, pin_memory=True, drop_last=True)
    vl_loader = DataLoader(ds_vl, batch_size=BATCH_SIZE * 2, shuffle=False, persistent_workers=True,
                           num_workers=2, pin_memory=True)
    te_loader = DataLoader(ds_te, batch_size=BATCH_SIZE * 2, shuffle=False, persistent_workers=True,
                           num_workers=2, pin_memory=True)

    print(f"✓ Dataset  train={len(ds_tr):,}  val={len(ds_vl):,}  test={len(ds_te):,}")
    print(f"  Input dim : {X_raw.shape[1]}  "
          f"(GTI mask channel idx={X_raw.shape[1]-1})")
    print(f"  Thresholds (norm) : C={thrs[0]:.3f}  M={thrs[1]:.3f}  X={thrs[2]:.3f}")
    focal_weights = compute_focal_weights(ds_tr)
    print(f"  Focal class weights: {focal_weights.numpy().round(3)}")
    return tr_loader, vl_loader, te_loader, (n_train, n_val, n_test), normalizer, focal_weights

print("✓ Dataset v4 defined  (dynamic gti_col, 5-tuple __getitem__)")


In [ ]:
import math


class CausalDilatedConv1d(nn.Module):
    """
    Single causal dilated 1D convolution with left-padding only.
    Ensures no future leakage: output[t] depends only on input[..t].
    """
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation   # left-pad only
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              dilation=dilation, padding=0)

    def forward(self, x):
        # x: [B, C, T]
        x = F.pad(x, (self.padding, 0))   # causal left-pad
        return self.conv(x)


class TCNResBlock(nn.Module):
    """
    Two-layer causal dilated conv with residual projection.
    Processes all input channels jointly → cross-detector noise rejection.
    """
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1  = CausalDilatedConv1d(in_ch,  out_ch, kernel_size, dilation)
        self.norm1  = nn.LayerNorm(out_ch)
        self.conv2  = CausalDilatedConv1d(out_ch, out_ch, kernel_size, dilation)
        self.norm2  = nn.LayerNorm(out_ch)
        self.drop   = nn.Dropout(dropout)
        self.act    = nn.GELU()
        # 1×1 projection if channel dims differ
        self.skip   = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        # x: [B, C_in, T]
        residual = self.skip(x)

        out = self.conv1(x)
        out = self.act(self.norm1(out.transpose(1, 2)).transpose(1, 2))
        out = self.drop(out)

        out = self.conv2(out)
        out = self.act(self.norm2(out.transpose(1, 2)).transpose(1, 2))
        out = self.drop(out)

        return out + residual   # residual connection


class MultiScaleTCNEncoder(nn.Module):
    def __init__(self, in_ch, channel_list, kernel_size, dilations, dropout):
        super().__init__()
        blocks = []
        ch = in_ch
        for out_ch in channel_list:               # depth-wise channel growth
            for d in dilations:                   # all dilations at each depth
                blocks.append(TCNResBlock(ch, out_ch, kernel_size, d, dropout))
                ch = out_ch
        self.blocks   = nn.ModuleList(blocks)
        self.out_ch   = ch

    def forward(self, x):
        # x: [B, C_in, T]
        for blk in self.blocks:
            x = blk(x)
        return x   # [B, C_out, T]



class MambaSSMBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv, expand):
        super().__init__()
        self.d_state  = d_state
        self.d_inner  = d_model * expand
        self.d_conv   = d_conv

        self.norm     = nn.LayerNorm(d_model)
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(self.d_inner, self.d_inner, d_conv,
                                  padding=d_conv - 1, groups=self.d_inner)
        self.x_proj   = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj  = nn.Linear(1, self.d_inner, bias=True)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.D        = nn.Parameter(torch.ones(self.d_inner))
        self.log_A    = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).unsqueeze(0).expand(self.d_inner, -1).clone())

    def _selective_scan(self, u, dt, A, B, C, D):
        """
        Numerically stable vectorised parallel scan.
        Clamps cumulative log-A to avoid exp overflow/underflow over long T.
        """
        # Discretise
        dA = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # [B, T, d, N]
        dB = dt.unsqueeze(-1) * B.unsqueeze(2)                           # [B, T, d, N]
        Bu = dB * u.unsqueeze(-1)                                         # [B, T, d, N]
    
        # Log-space cumulative product — CLAMP to prevent exp overflow
        log_dA     = torch.log(dA.clamp(min=1e-6))                       # clamp raised from 1e-30
        log_A_cum  = torch.cumsum(log_dA, dim=1).clamp(min=-30, max=30)  # ← key clamp
        A_cum      = torch.exp(log_A_cum)                                 # [B, T, d, N]
    
        inv_A_cum     = torch.exp(-log_A_cum)                            # [B, T, d, N]
        scaled_Bu     = inv_A_cum * Bu                                   # [B, T, d, N]
        cum_scaled_Bu = torch.cumsum(scaled_Bu, dim=1)                   # [B, T, d, N]
    
        h_seq = A_cum * cum_scaled_Bu                                    # [B, T, d, N]
        y_seq = (h_seq * C.unsqueeze(2)).sum(-1)                         # [B, T, d]
        return y_seq + u * D.unsqueeze(0).unsqueeze(0)

    def forward(self, x):
        # x: [B, T, d_model]
        residual = x
        x = self.norm(x)

        xz        = self.in_proj(x)
        x_in, z   = xz.chunk(2, dim=-1)                                  # [B, T, d_inner]

        x_conv    = self.conv1d(x_in.transpose(1, 2))[:, :, :x_in.size(1)].transpose(1, 2)
        x_conv    = F.silu(x_conv)                                       # [B, T, d_inner]

        bcdt      = self.x_proj(x_conv)                                  # [B, T, 2N+1]
        B_t       = bcdt[..., :self.d_state]
        C_t       = bcdt[..., self.d_state:2*self.d_state]
        dt_raw    = bcdt[..., 2*self.d_state:]

        dt        = F.softplus(self.dt_proj(dt_raw))                     # [B, T, d_inner]
        A         = -torch.exp(self.log_A)                               # [d_inner, N]

        y         = self._selective_scan(x_conv, dt, A, B_t, C_t, self.D)
        y         = y * F.silu(z)
        return self.out_proj(y) + residual


class ChannelGraphAttention(nn.Module):
    """
    Embeds each channel scalar → vector, then computes D×D 
    cross-channel attention at every timestep (PIDGMamba-style).
    """
    def __init__(self, in_dim, embed_dim=32):
        super().__init__()
        self.embed    = nn.Linear(1, embed_dim)         # scalar → vector per channel
        self.q        = nn.Linear(embed_dim, embed_dim)
        self.k        = nn.Linear(embed_dim, embed_dim)
        self.v        = nn.Linear(embed_dim, embed_dim)
        self.scale    = embed_dim ** 0.5
        self.out_proj = nn.Linear(embed_dim, 1)         # project back to scalar
        self.norm     = nn.LayerNorm(in_dim)

    def forward(self, x):
        # x: [B, T, D]
        B, T, D = x.shape
        xc   = self.embed(x.unsqueeze(-1))              # [B, T, D, embed_dim]
        Q    = self.q(xc)                               # [B, T, D, embed_dim]
        K    = self.k(xc)
        V    = self.v(xc)
        attn = torch.softmax(
            torch.matmul(Q, K.transpose(-1, -2)) / self.scale, dim=-1)  # [B, T, D, D]
        out  = torch.matmul(attn, V)                    # [B, T, D, embed_dim]
        out  = self.out_proj(out).squeeze(-1)           # [B, T, D]
        return self.norm(x + out)           # residual + norm



class SolarFlareTCNMamba(nn.Module):
    def __init__(self,
                 input_dim     = INPUT_DIM,
                 history       = HISTORY_STEPS,
                 forecast      = FORECAST_STEPS,
                 tcn_channels  = TCN_CHANNELS,
                 tcn_kernel    = TCN_KERNEL,
                 tcn_dilations = TCN_DILATIONS,
                 tcn_dropout   = TCN_DROPOUT,
                 mamba_d_model = MAMBA_D_MODEL,
                 mamba_d_state = MAMBA_D_STATE,
                 mamba_d_conv  = MAMBA_D_CONV,
                 mamba_expand  = MAMBA_EXPAND):
        super().__init__()
        self.forecast = forecast
        self.channel_attn = ChannelGraphAttention(in_dim=input_dim, embed_dim=32)
        # 1. Multi-scale dilated causal TCN — processes [B, D, T]
        self.tcn = MultiScaleTCNEncoder(
            in_ch=input_dim,
            channel_list=tcn_channels,
            kernel_size=tcn_kernel,
            dilations=tcn_dilations,
            dropout=tcn_dropout,
        )
        tcn_out_ch = self.tcn.out_ch   # = tcn_channels[-1] = 128

        # Project TCN output channels to Mamba d_model
        self.proj_to_mamba = nn.Linear(tcn_out_ch, mamba_d_model)

        # 2. Mamba SSM block — operates on [B, T, d_model]
        self.mamba = MambaSSMBlock(
            d_model=mamba_d_model,
            d_state=mamba_d_state,
            d_conv=mamba_d_conv,
            expand=mamba_expand,
        )

        # 3a. Classification head: last Mamba token → 4-class logit
        self.cls_head = nn.Sequential(
            nn.LayerNorm(mamba_d_model),
            nn.Linear(mamba_d_model, mamba_d_model // 2),
            nn.GELU(),
            nn.Dropout(tcn_dropout),
            nn.Linear(mamba_d_model // 2, 4),   # 0=Quiet, 1=C, 2=M, 3=X
        )

        # 3b. Regression head: pooled context → forecast sequence
        self.reg_head = nn.Sequential(
            nn.LayerNorm(mamba_d_model),
            nn.Linear(mamba_d_model, mamba_d_model),
            nn.GELU(),
            nn.Dropout(tcn_dropout),
            nn.Linear(mamba_d_model, forecast),
        )

        # Skip: project last observed value to full forecast window (residual)
        self.skip_proj = nn.Linear(1, forecast, bias=True)
        nn.init.constant_(self.skip_proj.weight, 1.0 / forecast)
        nn.init.zeros_(self.skip_proj.bias)

    def forward(self, x):
        # x: [B, T, D]
        B, T, D = x.shape
        x = self.channel_attn(x) 
        # TCN expects [B, D, T]
        h = self.tcn(x.permute(0, 2, 1))          # [B, C_tcn, T]
        h = h.permute(0, 2, 1)                     # [B, T, C_tcn]
        h = self.proj_to_mamba(h)                  # [B, T, d_model]

        # Mamba SSM — full sequence context
        h = self.mamba(h)                          # [B, T, d_model]

        # Use final token (most recent context) for heads
        ctx = h[:, -1, :]                          # [B, d_model]

        # Head A: 4-class flare probability
        cls_logits = self.cls_head(ctx)            # [B, 4]

        # Head B: regression with skip residual from last input value
        reg_out = self.reg_head(ctx)               # [B, F]
        last_val = x[:, -1, 0:1]                   # [B, 1]  last sx_log observation
        skip = self.skip_proj(last_val)            # [B, F]
        reg_pred = reg_out + skip                  # residual skip connection

        return reg_pred, cls_logits


print("✓ SolarFlareTCNMamba defined")
# _test_dim = INPUT_DIM if INPUT_DIM is not None else 7
# _demo = SolarFlareTCNMamba(input_dim=_test_dim)
# _x    = torch.randn(2, HISTORY_STEPS, INPUT_DIM)
# _r, _c = _demo(_x)
# print(f"  reg_pred  : {tuple(_r.shape)}   (expected [2, {FORECAST_STEPS}])")
# print(f"  cls_logits: {tuple(_c.shape)}   (expected [2, 4])")
# n_p = sum(p.numel() for p in _demo.parameters() if p.requires_grad)
# print(f"  Parameters: {n_p:,}")
# del _demo, _x, _r, _c


In [ ]:
class WingLoss(nn.Module):
    def __init__(self, w=WING_W, eps=WING_EPS):
        super().__init__()
        self.w   = w
        self.eps = eps
        self.C   = w - w * math.log(1 + w / eps)  # continuity constant

    def forward(self, pred, target, gti_mask=None, sample_weights=None):
        diff = pred - target                      # [B, F]
        abs_diff = diff.abs()

        # Wing piece-wise
        loss = torch.where(
            abs_diff < self.w,
            self.w * torch.log(1 + abs_diff / self.eps),
            abs_diff - self.C,
        )                                         # [B, F]

        # Asymmetric: penalise underprediction 2× (under-forecasting flares is dangerous)
        under_mask = (diff < 0).float()           # pred < target
        loss = loss * (1 + under_mask)            # [B, F]

        # GTI mask: zero out bad telemetry steps in the forecast window
        # We use a single mean mask across the history as a forecast-window proxy
        if gti_mask is not None:
            # gti_mask: [B, H] — average validity over input window
            validity = gti_mask.mean(dim=1, keepdim=True)  # [B, 1] → broadcast to [B, F]
            loss = loss * validity

        # Sample weights (flare-class oversample)
        if sample_weights is not None:
            sw = sample_weights / (sample_weights.mean() + 1e-8)
            loss = loss * sw.unsqueeze(1)

        return loss.mean()


class WeightedFocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA,
                 class_weights=None):
        super().__init__()
        self.gamma = gamma
        if class_weights is None:
            class_weights = torch.tensor([1.0, 4.0, 20.0, 60.0])
        self.register_buffer('cw', class_weights)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets,
                             weight=self.cw.to(logits.device),
                             reduction='none')     # [B]
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class MaskedPhysicsLoss(nn.Module):
    """
    Master loss combining Wing regression + Focal classification +
    asymmetric peak penalty, all controlled by gti_mask.
    """
    def __init__(self, w_reg=W_REG, w_class=W_CLASS, w_peak=W_PEAK,
                 w_delta=W_DELTA, w_curv=W_CURV, class_weights=None):
        super().__init__()
        self.w_reg   = w_reg
        self.w_class = w_class
        self.w_peak  = w_peak
        self.w_delta = w_delta   # first-difference (curve shape) loss weight
        self.w_curv  = w_curv    # second-difference (curvature) loss weight
        self.wing    = WingLoss()
        self.focal   = WeightedFocalLoss(class_weights=class_weights)

    def forward(self, reg_pred, target, cls_logits, cls_labels,
                gti_mask, sample_weights, x_input):
        # Boost loss for windows where the target actually curves (peak then fall).
        # This counteracts the quiet-period bias that causes linear extrapolation.
        true_peak_idx = target.argmax(dim=1)                      # [B]
        is_peaked = (true_peak_idx < int(FORECAST_STEPS * 0.70)).float()  # [B]
        peak_focal_w = 1.0 + (W_PEAK_FOCAL - 1.0) * is_peaked    # [B] — boost peaked windows
        # Merge with existing flare-class sample weights
        combined_sw = sample_weights * peak_focal_w               # [B]

        # 1. Wing regression loss with GTI masking + combined weights
        reg_loss = self.wing(reg_pred, target,
                             gti_mask=gti_mask,
                             sample_weights=combined_sw)

        # 2. Focal classification loss (no GTI masking — label is window-level)
        cls_loss = self.focal(cls_logits, cls_labels)

        # 3. Asymmetric peak penalty: extra push when predicted peak < actual peak
        pred_peak = reg_pred.max(dim=1).values     # [B]
        true_peak = target.max(dim=1).values       # [B]
        peak_loss = self.wing(pred_peak.unsqueeze(1),
                              true_peak.unsqueeze(1),
                              gti_mask=None,
                              sample_weights=combined_sw)

        # 4. Delta loss — penalise wrong curve SHAPE (first difference)
        # Forces the model to predict the correct gradient, not just magnitude.
        pred_delta = reg_pred[:, 1:] - reg_pred[:, :-1]    # [B, F-1]
        true_delta = target[:, 1:]   - target[:, :-1]      # [B, F-1]
        delta_loss = F.mse_loss(pred_delta, true_delta)

        # 5. Curvature loss — second difference, penalises wrong turning points
        pred_curv = pred_delta[:, 1:] - pred_delta[:, :-1]  # [B, F-2]
        true_curv = true_delta[:, 1:] - true_delta[:, :-1]  # [B, F-2]
        curv_loss = F.mse_loss(pred_curv, true_curv)

        # 6. Neupert physics: dSX/dt should correlate with HXR
        sx  = x_input[:, :, 0]                       # [B, H]  soft X-ray log
        hxr = x_input[:, :, 1:4].mean(dim=-1)        # [B, H]  mean HX log
        d_sx  = torch.diff(sx,  dim=1) / CADENCE_S   # [B, H-1]
        d_hxr = torch.diff(hxr, dim=1) / CADENCE_S   # [B, H-1]
        neupert_loss = F.mse_loss(
            F.normalize(d_sx,  dim=1),
            F.normalize(d_hxr, dim=1)
        )

        total = (self.w_reg   * reg_loss   +
                 self.w_class * cls_loss   +
                 self.w_peak  * peak_loss  +
                 self.w_delta * delta_loss +
                 self.w_curv  * curv_loss  +
                 W_NEUPERT    * neupert_loss)

        return total, {
            'reg'    : reg_loss.item(),
            'cls'    : cls_loss.item(),
            'peak'   : peak_loss.item(),
            'delta'  : delta_loss.item(),
            'curv'   : curv_loss.item(),
            'neupert': neupert_loss.item(),
        }


print("✓ MaskedPhysicsLoss defined  (Wing + Focal + AsymPeak + Delta + Curv + GTI-masking)")
print(f"  Wing params  : w={WING_W}  eps={WING_EPS}")
print(f"  Focal γ      : {FOCAL_GAMMA}  |  class weights [Q=1, C=4, M=20, X=60]")
print(f"  Loss weights : reg={W_REG}  cls={W_CLASS}  peak={W_PEAK}  delta={W_DELTA}  curv={W_CURV}")
print(f"  Peak-focal   : W_PEAK_FOCAL={W_PEAK_FOCAL}  (extra weight on turning-point windows)")
print(f"  EMA alpha    : {EMA_ALPHA}  ({'enabled' if EMA_ALPHA > 0 else 'disabled'})")


In [ ]:
def run_epoch_v4(model, loader, criterion, optimizer, scheduler,
                 scaler, is_train, device):
    model.train() if is_train else model.eval()
    tot, n = 0.0, 0
    comp_sums = {'reg': 0., 'cls': 0., 'peak': 0., 'delta': 0., 'curv': 0., 'neupert': 0.}
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y, gti_mask, wt, lbl in loader:
            x        = x.to(device,   non_blocking=True)  # [B, H, D]
            y        = y.to(device,   non_blocking=True)  # [B, F]
            gti_mask = gti_mask.to(device, non_blocking=True)  # [B, H]
            wt       = wt.to(device,  non_blocking=True)  # [B]
            lbl      = lbl.to(device, non_blocking=True)  # [B]

            # fp16 forward pass via T4 Tensor Cores
            with autocast():
                reg_pred, cls_logits = model(x)
                loss, comp = criterion(reg_pred, y, cls_logits,
                                       lbl, gti_mask, wt, x)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()   # OneCycleLR: step per batch

            bs = x.size(0)
            tot += loss.item() * bs
            n   += bs
            for k in comp_sums:
                comp_sums[k] += comp[k] * bs

    return tot / n, {k: v / n for k, v in comp_sums.items()}


def train_model_v4(model, tr_loader, vl_loader, criterion, epochs=EPOCHS):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # OneCycleLR: warm-up + cosine annealing, steps per batch
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        epochs=epochs, steps_per_epoch=len(tr_loader),
        pct_start=0.15, anneal_strategy='cos',
        div_factor=10, final_div_factor=100,
    )
    scaler     = GradScaler()
    best_val   = 1e9
    patience   = 0
    max_patience = 10
    best_state = None
    history    = {'train': [], 'val': [], 'lr': []}

    fig = plt.figure(figsize=(16, 9))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)
    ax_loss = fig.add_subplot(gs[0, 0])
    ax_lr   = fig.add_subplot(gs[0, 1])
    ax_comp = fig.add_subplot(gs[0, 2])
    ax_vbar = fig.add_subplot(gs[1, :])
    COLORS  = {'reg': '#4C72B0', 'cls': '#55A868', 'peak': '#C44E52'}

    start = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_comp = run_epoch_v4(
            model, tr_loader, criterion, optimizer,
            scheduler, scaler, True, device)

        # Validation (no scheduler step)
        model.eval()
        vl_tot, vl_n = 0.0, 0
        vl_cs = {'reg': 0., 'cls': 0., 'peak': 0.}
        with torch.no_grad():
            for x, y, gti_mask, wt, lbl in vl_loader:
                x        = x.to(device)
                y        = y.to(device)
                gti_mask = gti_mask.to(device)
                wt       = wt.to(device)
                lbl      = lbl.to(device)
                with autocast():
                    rp, cl = model(x)
                    loss, comp = criterion(rp, y, cl, lbl, gti_mask, wt)
                bs = x.size(0)
                vl_tot += loss.item() * bs
                vl_n   += bs
                for k in vl_cs: vl_cs[k] += comp[k] * bs
        vl_loss = vl_tot / vl_n
        vl_comp = {k: v / vl_n for k, v in vl_cs.items()}

        lr_now = optimizer.param_groups[0]['lr']
        history['train'].append(tr_loss)
        history['val'].append(vl_loss)
        history['lr'].append(lr_now)

        marker = ''
        if vl_loss < best_val:
            best_val   = vl_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
            marker     = '★'
        else:
            patience += 1

        print(f"Ep {epoch:3d}/{epochs} | tr={tr_loss:.4f}  vl={vl_loss:.4f} {marker} "
              f"| reg={tr_comp['reg']:.3f} cls={tr_comp['cls']:.3f} "
              f"peak={tr_comp['peak']:.3f} delta={tr_comp.get('delta',0):.3f} "
              f"curv={tr_comp.get('curv',0):.3f} npt={tr_comp['neupert']:.3f} "
              f"| lr={lr_now:.2e}  pat={patience}  {time.time()-start:.0f}s")
        if epoch % 5 == 0 or epoch == epochs:
            ep_ax = list(range(1, epoch + 1))
            for ax in [ax_loss, ax_lr, ax_comp, ax_vbar]: ax.cla()

            ax_loss.plot(ep_ax, history['train'], lw=2, color='#2196F3', label='Train')
            ax_loss.plot(ep_ax, history['val'],   lw=2, color='#F44336', label='Val')
            ax_loss.set(title='Total Loss', xlabel='Epoch', ylabel='Loss')
            ax_loss.legend(fontsize=9); ax_loss.grid(alpha=0.3)

            ax_lr.plot(ep_ax, history['lr'], lw=2, color='#4CAF50')
            ax_lr.set_yscale('log')
            ax_lr.set(title='LR (OneCycle)', xlabel='Epoch', ylabel='LR')
            ax_lr.grid(alpha=0.3)

            comps  = ['reg', 'cls', 'peak', 'neupert']
            ws     = [W_REG, W_CLASS, W_PEAK, W_NEUPERT]
            vals   = [tr_comp[k] * w for k, w in zip(comps, ws)]
            bars   = ax_comp.bar(comps, vals, color=[COLORS[c] for c in comps],
                                 edgecolor='white', linewidth=0.8)
            for bar, val in zip(bars, vals):
                ax_comp.text(bar.get_x() + bar.get_width()/2,
                             bar.get_height() + 0.0001,
                             f'{val:.4f}', ha='center', va='bottom', fontsize=8)
            ax_comp.set(title=f'Weighted Components (Ep {epoch})', ylabel='Weighted value')
            ax_comp.grid(alpha=0.3, axis='y')

            ep_marks = list(range(5, epoch + 1, 5)) + ([epoch] if epoch % 5 else [])
            vl_marks = [history['val'][e - 1] for e in ep_marks]
            bc = ['#F44336' if v == min(vl_marks) else '#90CAF9' for v in vl_marks]
            ax_vbar.bar(ep_marks, vl_marks, color=bc, width=3, edgecolor='white')
            ax_vbar.axhline(best_val, color='red', lw=1.5, ls='--',
                            label=f'Best={best_val:.4f}')
            ax_vbar.set(title='Validation Loss per Checkpoint',
                        xlabel='Epoch', ylabel='Val Loss')
            ax_vbar.legend(fontsize=9); ax_vbar.grid(alpha=0.3, axis='y')

            fig.suptitle(f'TCN-Mamba Training — Epoch {epoch}/{epochs}',
                         fontsize=13, fontweight='bold')
            clear_output(wait=True); display(fig); plt.pause(0.01)

        if patience >= max_patience:
            print(f"  ↳ Early stopping at epoch {epoch}  (best val={best_val:.4f})")
            break

    plt.close(fig)
    if best_state:
        model.load_state_dict(best_state)
    print(f"\n✓ Training complete  |  best_val={best_val:.4f}")
    return model, history


print("✓ Training loop v4 defined  (AMP + DataParallel + OneCycleLR-per-batch)")


In [ ]:
def classify_alert(peak_physical):
    if   peak_physical >= X_THRESHOLD: return 'X-Class (Severe)',   '#8B0000'
    elif peak_physical >= M_THRESHOLD: return 'M-Class (Moderate)', '#FF4500'
    elif peak_physical >= C_THRESHOLD: return 'C-Class (Minor)',    '#FFA500'
    else:                              return 'Quiet (No Impact)',   '#2196F3'

def _log_r2(actual, pred):
    ss_res = np.sum((actual - pred) ** 2)
    ss_tot = np.sum((actual - actual.mean()) ** 2)
    return float(1 - ss_res / (ss_tot + 1e-12))


@torch.no_grad()
def plot_forecast_grid(model, dataset, normalizer, n_samples=6,
                       cadence_s=CADENCE_S, title='Forecast vs Actual',
                       indices=None, mode='random',
                       all_true_cls=None, all_pred_cls=None,
                       all_true_norm=None, all_pred_norm=None):
    """
    mode: 'random' | 'worst' | 'missed' | 'false_alarm' | 'best' | 'flares_only'
    Pass all_true_cls, all_pred_cls, all_true_norm, all_pred_norm for error-based modes.
    """
    model.eval()
    N = len(dataset)

    if indices is not None:
        base_idxs = list(indices[:n_samples])

    elif mode == 'random':
        base_idxs = np.random.choice(N, n_samples, replace=False).tolist()
        # optionally still force 1-2 flare peaks in the mix:
        peaks = [dataset[i][1].max().item() for i in range(N)]
        sorted_idx = np.argsort(peaks)[::-1]
        base_idxs[-1] = int(sorted_idx[0])
        base_idxs[-2] = int(sorted_idx[1])

    elif mode == 'flares_only':
        thrs_n = normalizer.log1p_thresholds()
        flare_idx = [i for i in range(N) if dataset[i][1].max().item() >= thrs_n[0]]
        base_idxs = np.random.choice(flare_idx, size=min(n_samples, len(flare_idx)), replace=False).tolist()

    elif mode == 'missed' and all_true_cls is not None:
        # True M/X but model predicted Quiet or C
        idx = np.where((all_true_cls >= 2) & (all_pred_cls <= 1))[0]
        base_idxs = np.random.choice(idx, size=min(n_samples, len(idx)), replace=False).tolist() if len(idx) else []

    elif mode == 'false_alarm' and all_true_cls is not None:
        # True Quiet but model predicted M/X
        idx = np.where((all_true_cls == 0) & (all_pred_cls >= 2))[0]
        base_idxs = np.random.choice(idx, size=min(n_samples, len(idx)), replace=False).tolist() if len(idx) else []

    elif mode == 'worst' and all_true_norm is not None:
        errors = np.abs(all_pred_norm.max(axis=1) - all_true_norm.max(axis=1))
        base_idxs = np.argsort(errors)[-n_samples:][::-1].tolist()

    elif mode == 'best' and all_true_norm is not None:
        errors = np.abs(all_pred_norm.max(axis=1) - all_true_norm.max(axis=1))
        flare_mask = all_true_cls >= 1
        flare_errors = np.where(flare_mask, errors, np.inf)
        base_idxs = np.argsort(flare_errors)[:n_samples].tolist()

    else:
        base_idxs = np.random.choice(N, n_samples, replace=False).tolist()

    ncols, nrows = 3, (n_samples + 2) // 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4))
    axes = np.array(axes).flatten()
    H, FCAST       = HISTORY_STEPS, FORECAST_STEPS
    t_hist     = np.arange(H) * cadence_s
    t_forecast = np.arange(H, H + FCAST) * cadence_s

    for ax_i, idx in enumerate(base_idxs):
        ax = axes[ax_i]
        x, y_norm, *_ = dataset[idx]
        x_batch = x.unsqueeze(0).to(device)

        pred_norm, cls_logits = model(x_batch)
        pred_norm = pred_norm.squeeze(0).cpu().numpy()
        y_norm_np = y_norm.numpy()

        pred_phys = normalizer.inverse_target(pred_norm)
        act_phys  = normalizer.inverse_target(y_norm_np)
        hist_phys = normalizer.inverse_target(x[:, 0].numpy())

        r2 = _log_r2(y_norm_np, pred_norm)
        pred_alert, pred_color = classify_alert(pred_phys.max())
        act_alert,  act_color  = classify_alert(act_phys.max())

        ax.plot(t_hist,     hist_phys, 'b-',  lw=1.5, label='Input history')
        ax.plot(t_forecast, act_phys,  'g-',  lw=2,   label='Actual future')
        ax.plot(t_forecast, pred_phys, 'r--', lw=2,   label='Predicted future')
        ax.fill_between(t_forecast, act_phys, pred_phys,
                        alpha=0.12, color='red', label='Mismatch margin')
        ax.axvline(t_hist[-1], color='gray', ls=':', lw=1.5)
        ax.axhline(C_THRESHOLD, color='orange', ls='--', alpha=0.6, lw=1,
                   label=f'C ({C_THRESHOLD:.0f})')
        ax.axhline(M_THRESHOLD, color='purple', ls='--', alpha=0.6, lw=1,
                   label=f'M ({M_THRESHOLD:.0f})')
        ax.set_yscale('log'); ax.set_ylim(bottom=0.5)
        ax.set_xlabel('Time (s)'); ax.set_ylabel('SoLEXS (counts/s)')
        ax.set_title(f'Sample #{idx} | LogR²={r2:.3f}\nActual: {act_alert}  Pred: {pred_alert}', fontsize=9)
        ax.legend(fontsize=7, loc='upper left'); ax.grid(True, which='both', alpha=0.3)

    for j in range(len(base_idxs), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f'{title} [{mode.upper()}]', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    return fig


@torch.no_grad()
def run_custom_inference(model, df_full, normalizer,
                         sample_idx=None, start_time_str=None,
                         history_steps=HISTORY_STEPS, forecast_steps=FORECAST_STEPS,
                         cadence_s=CADENCE_S):
    """
    Run inference on a specific time-window and show the alert-level comparison plot.
    df_full must already have been processed with build_features_v4().
    """
    model.eval()
    N     = len(df_full)
    total = history_steps + forecast_steps

    if start_time_str is not None:
        try:
            target_dt  = pd.to_datetime(start_time_str).tz_localize(None)
            sample_idx = df_full.index.get_indexer([target_dt], method='nearest')[0]
            print(f"  Aligned '{start_time_str}' → index {sample_idx} ({df_full.index[sample_idx]})")
        except Exception as e:
            print(f"  parse error: {e}, using random index")
            sample_idx = None

    if sample_idx is None:
        sample_idx = np.random.randint(0, N - total)
        print(f"  Random index: {sample_idx}  ({df_full.index[sample_idx]})")
    else:
        sample_idx = min(max(0, sample_idx), N - total)

    window   = df_full.iloc[sample_idx : sample_idx + total]
    X_raw    = window[FEATURE_COLS_V4].values.astype(np.float32)   # uses dynamic global
    X_norm   = normalizer.transform(X_raw)

    x_in     = torch.tensor(X_norm[:history_steps]).unsqueeze(0).to(device)
    pred_norm, cls_log = model(x_in)
    pred_norm = pred_norm.squeeze(0).cpu().numpy()

    target_raw  = window['target'].values.astype(np.float32)
    target_norm = (target_raw - normalizer.mean_[0]) / normalizer.std_[0]

    hist_phys = normalizer.inverse_target(target_norm[:history_steps])
    act_phys  = normalizer.inverse_target(target_norm[history_steps:])
    pred_phys = normalizer.inverse_target(pred_norm)

    t_hist = np.arange(history_steps) * cadence_s
    t_fore = np.arange(history_steps, history_steps + forecast_steps) * cadence_s

    pred_alert, pred_color = classify_alert(pred_phys.max())
    act_alert,  act_color  = classify_alert(act_phys.max())
    r2 = _log_r2(target_norm[history_steps:], pred_norm)

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(t_hist, hist_phys, 'b-',  lw=2,   label=f'Input History')
    ax.plot(t_fore, act_phys,  'g-',  lw=2.5, label='Actual Future Target')
    ax.plot(t_fore, pred_phys, 'r--', lw=2.5, label='Predicted Future Target')
    ax.fill_between(t_fore, act_phys, pred_phys, alpha=0.15, color='red',
                    label='Forecast Mismatch Margin')
    ax.axvline(t_hist[-1], color='gray', ls=':', lw=1.5, label='Forecast Start')
    ax.axhline(C_THRESHOLD, color='orange', ls='--', alpha=0.7, lw=1.5,
               label=f'C-class ({C_THRESHOLD:.1f})')
    ax.axhline(M_THRESHOLD, color='purple', ls='--', alpha=0.7, lw=1.5,
               label=f'M-class ({M_THRESHOLD:.1f})')
    ax.set_yscale('log'); ax.set_ylim(bottom=0.5)
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_ylabel('SoLEXS Count Rate (counts/s)', fontsize=11)
    ax.set_title(
        f'Custom Inference  |  Index: {sample_idx}\\n'
        f'Actual: {act_alert} (Peak={act_phys.max():.1f})  |  '
        f'Predicted: {pred_alert} (Peak={pred_phys.max():.1f})  |  LogR²={r2:.3f}',
        fontsize=11, fontweight='bold'
    )
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); plt.show()
    return fig

print("✓ Visualisation helpers defined")

In [ ]:
@torch.no_grad()
def evaluate_classification(model, loader, normalizer, device):
    """
    Compute confusion matrix + TSS + precision/recall for each class.
    """
    model.eval()
    all_pred, all_true = [], []
    thrs = normalizer.log1p_thresholds()

    for x, y, *_ in loader:      # *_ absorbs gti_mask, wt, lbl safely
        x        = x.to(device)
        pred_norm, _ = model(x)
        pred_norm = pred_norm.cpu().numpy()
        y_np      = y.numpy()

        for i in range(len(pred_norm)):
            peak_p = pred_norm[i].max()
            peak_t = y_np[i].max()
            def cls(v):
                if   v >= thrs[2]: return 3
                elif v >= thrs[1]: return 2
                elif v >= thrs[0]: return 1
                else:              return 0
            all_pred.append(cls(peak_p))
            all_true.append(cls(peak_t))

    class_names = ['Quiet', 'C', 'M', 'X']
    n_cls = len(class_names)
    cm    = np.zeros((n_cls, n_cls), dtype=int)
    for t, p in zip(all_true, all_pred):
        cm[t][p] += 1

    print("\\n── Confusion Matrix (rows=true, cols=pred) ───────────────────────────")
    header = '          ' + ' '.join(f'{c:>8}' for c in class_names)
    print(header)
    for i, row_name in enumerate(class_names):
        print(f'  {row_name:>6}  ' + ' '.join(f'{cm[i,j]:>8}' for j in range(n_cls)))

    print("\\n── Per-class Metrics ────────────────────────────────────────────────")
    tss_list = []
    for c in range(n_cls):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        TPR = TP / (TP + FN + 1e-9)
        TNR = TN / (TN + FP + 1e-9)
        TSS = TPR + TNR - 1
        PRE = TP / (TP + FP + 1e-9)
        tss_list.append(TSS)
        print(f'  {class_names[c]:>6} | TPR={TPR:.3f}  TNR={TNR:.3f}  Prec={PRE:.3f}  TSS={TSS:.3f}')

    print(f"\\n  Mean TSS (macro) = {np.mean(tss_list):.3f}")
    print(f"  Overall accuracy = {np.diag(cm).sum() / cm.sum():.3f}")
    return {'cm': cm, 'tss': tss_list}

print("✓ Evaluation helpers defined")

In [ ]:
print("=" * 65)
print("  SOLAR FLARE FORECASTING — TCN-Mamba v4  |  Aditya-L1")
print("=" * 65)

print("\\n[1/6] Loading data …")
try:
    df_raw = load_payload_data(SOLEXS_DIR, HELIOS_DIR, cadence='10S')
    print("  ✓ Real Aditya-L1 data loaded")
    DATA_SOURCE = 'real'
except Exception as exc:
    print(f"  ⚠  FITS load failed ({exc})")
    print("  ↳  Using calibrated synthetic data (60 days, 3 HEL1OS detectors)")
    df_raw = create_synthetic_solar_data(n_days=60)
    DATA_SOURCE = 'synthetic'

df_raw = df_raw.replace([np.inf, -np.inf], np.nan).dropna()
print(f"  Rows: {len(df_raw):,}  |  source: {DATA_SOURCE}")
hx_cols = sorted(c for c in df_raw.columns if c.startswith('hx_'))
print(f"  HEL1OS detectors in data: {hx_cols}")

print("\\n[1b] Loading GTI mask …")
if DATA_SOURCE == 'real':
    try:
        gti_mask = build_gti_mask(GTI_DIR, df_raw.index)
    except Exception as e:
        print(f"  ⚠  GTI load error ({e}); mask=1.0")
        gti_mask = pd.Series(np.ones(len(df_raw), np.float32), index=df_raw.index)
else:
    # Synthetic data has no bad intervals — generate a realistic-looking mask
    # (90% valid with a few short dropout windows, to exercise the GTI path)
    _mask = np.ones(len(df_raw), dtype=np.float32)
    rng_g = np.random.default_rng(SEED + 99)
    for _ in range(40):
        start = rng_g.integers(0, len(_mask) - 100)
        length = rng_g.integers(5, 60)
        _mask[start:start + length] = 0.0
    gti_mask = pd.Series(_mask, index=df_raw.index)
    n_bad = int((1 - _mask).sum())
    print(f"  Synthetic GTI: {n_bad} dropout steps ({100*n_bad/len(_mask):.1f}%) injected")

print("\\n[2/6] Building features …")
df_feat, FEATURE_COLS_V4 = build_features_v4(df_raw, gti_mask_series=gti_mask)
INPUT_DIM = len(FEATURE_COLS_V4)
n_det = len([c for c in FEATURE_COLS_V4 if c.startswith('hx_log_')])
print(f"  HEL1OS detectors  : {n_det}  →  INPUT_DIM = {INPUT_DIM}  (= 4 + 3×{n_det})")
print(f"  Feature columns   : {FEATURE_COLS_V4}")
print(f"  GTI valid steps   : {int(df_feat['gti_mask'].sum()):,}/{len(df_feat):,}")

fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)
axes[0].semilogy(df_feat.index, df_raw['solexs_rate'], lw=0.5, color='darkorange')
axes[0].axhline(C_THRESHOLD, ls='--', color='orange', lw=1, label=f'C ({C_THRESHOLD})')
axes[0].axhline(M_THRESHOLD, ls='--', color='red',    lw=1, label=f'M ({M_THRESHOLD})')
axes[0].set_ylabel('SoLEXS (counts/s)'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].set_title('Aditya-L1 Light Curves & GTI mask', fontweight='bold')

colors_hx = ['mediumpurple', 'cornflowerblue', 'teal']
for i, col in enumerate(hx_cols):
    axes[1].semilogy(df_feat.index, df_raw[col].clip(lower=0.01), lw=0.4,
                     color=colors_hx[i % len(colors_hx)], alpha=0.8, label=col)
axes[1].set_ylabel('HEL1OS (counts/s)'); axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

axes[2].fill_between(df_feat.index, df_feat['gti_mask'], alpha=0.4,
                     color='green', label='GTI valid (1.0)')
axes[2].plot(df_feat.index, df_feat['gti_mask'], lw=0.3, color='green')
axes[2].set_ylabel('GTI mask'); axes[2].set_xlabel('Time')
axes[2].set_ylim(-0.05, 1.15); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\\n[3/6] Creating datasets …")
normalizer = FeatureNormalizer()
tr_loader, vl_loader, te_loader, split_idx, normalizer, focal_weights = make_loaders_v4(df_feat, normalizer, feature_cols=FEATURE_COLS_V4)

print("\n[4/6] Building model …")
model = SolarFlareTCNMamba(input_dim=INPUT_DIM).to(device)
# if NUM_GPUS > 1:
#     model = nn.DataParallel(model)
#     print(f"  Wrapping in nn.DataParallel ({NUM_GPUS} GPUs)")
import torch._dynamo
torch._dynamo.config.cache_size_limit = 64 
model = torch.compile(model, mode='default', fullgraph=False, dynamic=True) #########################################

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {n_params:,}  (INPUT_DIM={INPUT_DIM})")
criterion = MaskedPhysicsLoss(class_weights=focal_weights.to(device))

# with torch.no_grad():
#     dummy = torch.randn(4, HISTORY_STEPS, INPUT_DIM).to(device)
#     _ = model(dummy)
# print("  ✓ Compile warmup done")

print(f"\n[5/6] Training ({EPOCHS} epochs, OneCycleLR per batch) …")

def save_checkpoint(model, normalizer, history, epoch, path):
    torch.save({
        'model_state'     : model.state_dict(),
        'normalizer_mean' : normalizer.mean_,
        'normalizer_std'  : normalizer.std_,
        'history'         : history,
        'architecture'    : 'TCNMamba-v4',
        'feature_cols'    : FEATURE_COLS_V4,
        'input_dim'       : INPUT_DIM,
        'epoch'           : epoch,
    }, path)

ckpt_path  = '/kaggle/working/solar_flare_tcnmamba_best.pt'
latest_path = '/kaggle/working/solar_flare_tcnmamba_latest.pt'

best_val   = float('inf')
history    = {'tr_loss': [], 'vl_loss': [], 'tr_comp': [], 'vl_comp': []}
t0_total   = time.time()

try:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        steps_per_epoch=len(tr_loader), epochs=EPOCHS)
    scaler = GradScaler()

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        tr_loss, tr_comp = run_epoch_v4(model, tr_loader, criterion, optimizer,
                                         scheduler, scaler, True, device)
        vl_loss, vl_comp = run_epoch_v4(model, vl_loader, criterion, None,
                                         None, scaler, False, device)

        history['tr_loss'].append(tr_loss)
        history['vl_loss'].append(vl_loss)
        history['tr_comp'].append(tr_comp)
        history['vl_comp'].append(vl_comp)

        elapsed   = time.time() - t0
        remaining = elapsed * (EPOCHS - epoch)
        lr_now    = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch:02d}/{EPOCHS}"
              f"  │  tr={tr_loss:.4f}  vl={vl_loss:.4f}"
              f"  │  reg={tr_comp.get('reg',0):.4f}"
              f"  cls={tr_comp.get('cls',0):.4f}"
              f"  peak={tr_comp.get('peak',0):.4f}"
              f"  delta={tr_comp.get('delta',0):.4f}"
              f"  curv={tr_comp.get('curv',0):.4f}"
              f"  │  lr={lr_now:.2e}"
              f"  │  {elapsed:.0f}s/epoch  ETA={remaining/60:.1f}min")

        save_checkpoint(model, normalizer, history, epoch, latest_path)

        if vl_loss < best_val:
            best_val = vl_loss
            save_checkpoint(model, normalizer, history, epoch, ckpt_path)
            print(f"    ✓ Best model saved (val={best_val:.4f})")

except KeyboardInterrupt:
    print("\n  ⚠  Interrupted — saving current state …")
    save_checkpoint(model, normalizer, history, epoch, latest_path)
    print(f"  ✓ Checkpoint saved → {latest_path}")
    print(f"  ✓ Best so far      → {ckpt_path}  (val={best_val:.4f})")

total_time = (time.time() - t0_total) / 60
print(f"\n  Training complete in {total_time:.1f} min")
print(f"  Best val loss : {best_val:.4f}")
print(f"  Best ckpt     : {ckpt_path}")
print(f"  Latest ckpt   : {latest_path}")

In [ ]:
model_inner = model._orig_mod  # unwrap compiled model for variable batch sizes

# Option A: Random sample
fig_a = run_custom_inference(
    model_inner, df_feat, normalizer,
    sample_idx=None, start_time_str=None,
)

# Option B: Largest flare window
peak_idx = int(df_raw['solexs_rate'].values.argmax())
peak_idx  = max(0, peak_idx - HISTORY_STEPS)
print(f"\nLargest flare at step {peak_idx} ({df_feat.index[peak_idx] if peak_idx < len(df_feat) else 'OOB'})")
if peak_idx + HISTORY_STEPS + FORECAST_STEPS < len(df_feat):
    fig_b = run_custom_inference(model_inner, df_feat, normalizer, sample_idx=peak_idx)

In [ ]:
import torch._dynamo

def load_checkpoint_v4(ckpt_path='/kaggle/input/datasets/garvsaluja/models/solar_flare_tcnmamba_bestest.pt'):
    global FEATURE_COLS_V4, INPUT_DIM

    # Clear any stale compiled graphs from training session
    torch._dynamo.reset()
    torch.cuda.empty_cache()

    # Load to CPU first — avoids GPU memory corruption on reload
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    # Restore dynamic feature config
    FEATURE_COLS_V4 = ckpt.get('feature_cols', FEATURE_COLS_V4)
    INPUT_DIM       = ckpt.get('input_dim', len(FEATURE_COLS_V4))

    # Build raw model — no compile, no DataParallel for eval
    _model = SolarFlareTCNMamba(input_dim=INPUT_DIM)

    # Strip compile/DataParallel prefixes from saved keys
    state = {
        k.replace('module.', '').replace('_orig_mod.', ''): v.clone()
        for k, v in ckpt['model_state'].items()
    }
    _model.load_state_dict(state, strict=True)

    # Move to GPU as plain model — no DataParallel, no torch.compile
    _model = _model.to(device).eval()

    # Restore normalizer
    _norm       = FeatureNormalizer()
    _norm.mean_ = ckpt['normalizer_mean']
    _norm.std_  = ckpt['normalizer_std']

    arch = ckpt.get('architecture', 'TCNMamba-v4')
    print(f"✓ Checkpoint loaded ({arch}) from {ckpt_path}")
    print(f"  INPUT_DIM       : {INPUT_DIM}")
    print(f"  FEATURE_COLS_V4 : {FEATURE_COLS_V4}")
    return _model, _norm

model, normalizer = load_checkpoint_v4()

# Separate eval reference — never wrap this in compile/DataParallel
model_eval  = model
model_inner = model

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
torch.cuda.empty_cache()
torch.cuda.synchronize()
import numpy as np
import torch
from sklearn.metrics import confusion_matrix, classification_report

ckpt_path = '/kaggle/input/datasets/garvsaluja/models/solar_flare_tcnmamba_bestest.pt'
model_eval, normalizer_eval = load_checkpoint_v4(ckpt_path)
model_eval.eval()
print(f"✓ Model loaded for evaluation")

X_all = normalizer_eval.transform(df_feat[FEATURE_COLS_V4].values.astype(np.float32))
y_all = ((df_feat['target'].values.astype(np.float32) - normalizer_eval.mean_[0])
          / normalizer_eval.std_[0])
thrs_n = normalizer_eval.log1p_thresholds()

N       = len(X_all)
n_test  = max(HISTORY_STEPS + FORECAST_STEPS + 1, int(N * 0.05))
n_val   = max(HISTORY_STEPS + FORECAST_STEPS + 1, int(N * 0.15))
n_train = N - n_val - n_test

te_ds     = SolarFlareDataset(X_all[n_train+n_val:], y_all[n_train+n_val:],
                               thresholds_norm=thrs_n)
te_loader = DataLoader(te_ds, batch_size=64, shuffle=False,
                       num_workers=2, pin_memory=True)

print(f"  Test samples : {len(te_ds):,}")

all_pred_norm, all_true_norm = [], []
all_pred_cls,  all_true_cls  = [], []

t_C, t_M, t_X = thrs_n

def classify(peak):
    if   peak >= t_X: return 3
    elif peak >= t_M: return 2
    elif peak >= t_C: return 1
    else:             return 0

with torch.no_grad():
    for x, y, gti_mask, wt, lbl in te_loader:
        x = x.to(device)
        pred_norm, _ = model_eval(x)
        pred_norm = pred_norm.cpu().numpy()
        y_np      = y.numpy()

        all_pred_norm.append(pred_norm)
        all_true_norm.append(y_np)

        for i in range(len(pred_norm)):
            all_pred_cls.append(classify(pred_norm[i].max()))
            all_true_cls.append(classify(y_np[i].max()))

all_pred_norm = np.concatenate(all_pred_norm, axis=0)   # [N_te, F]
all_true_norm = np.concatenate(all_true_norm, axis=0)   # [N_te, F]
all_pred_cls  = np.array(all_pred_cls)
all_true_cls  = np.array(all_true_cls)

mae  = np.abs(all_pred_norm - all_true_norm).mean()
rmse = np.sqrt(((all_pred_norm - all_true_norm) ** 2).mean())
ss_res = ((all_true_norm - all_pred_norm) ** 2).sum()
ss_tot = ((all_true_norm - all_true_norm.mean()) ** 2).sum()
r2     = 1 - ss_res / (ss_tot + 1e-12)

# Peak-level regression
pred_peaks = all_pred_norm.max(axis=1)
true_peaks = all_true_norm.max(axis=1)
peak_mae   = np.abs(pred_peaks - true_peaks).mean()
peak_corr  = np.corrcoef(pred_peaks, true_peaks)[0, 1]

print("\n── Regression Metrics ───────────────────────────────────────────────")
print(f"  MAE  (sequence) : {mae:.4f}")
print(f"  RMSE (sequence) : {rmse:.4f}")
print(f"  R²   (sequence) : {r2:.4f}")
print(f"  Peak MAE        : {peak_mae:.4f}")
print(f"  Peak Correlation: {peak_corr:.4f}")

class_names = ['Quiet', 'C', 'M', 'X']
cm = confusion_matrix(all_true_cls, all_pred_cls, labels=[0,1,2,3])

print("\n── Confusion Matrix (rows=true, cols=pred) ──────────────────────────")
header = '          ' + ' '.join(f'{c:>8}' for c in class_names)
print(header)
for i, name in enumerate(class_names):
    print(f'  {name:>6}  ' + ' '.join(f'{cm[i,j]:>8}' for j in range(4)))

print("\n── Per-class Metrics ────────────────────────────────────────────────")
tss_list = []
for c in range(4):
    TP = cm[c, c]
    FN = cm[c, :].sum() - TP
    FP = cm[:, c].sum() - TP
    TN = cm.sum() - TP - FN - FP
    TPR = TP / (TP + FN + 1e-9)   # sensitivity / recall
    TNR = TN / (TN + FP + 1e-9)   # specificity
    PRE = TP / (TP + FP + 1e-9)   # precision
    F1  = 2 * PRE * TPR / (PRE + TPR + 1e-9)
    TSS = TPR + TNR - 1            # true skill statistic (key solar metric)
    HSS = (2 * (TP*TN - FN*FP)) / ((TP+FN)*(FN+TN) + (TP+FP)*(FP+TN) + 1e-9)
    tss_list.append(TSS)
    print(f"  {class_names[c]:>6} | TPR={TPR:.3f}  TNR={TNR:.3f}  "
          f"Prec={PRE:.3f}  F1={F1:.3f}  TSS={TSS:.3f}  HSS={HSS:.3f}")

overall_acc = np.diag(cm).sum() / cm.sum()
mean_tss    = np.mean(tss_list)
print(f"\n  Mean TSS (macro) : {mean_tss:.4f}  "
      f"({'good' if mean_tss > 0.5 else 'needs improvement'})")
print(f"  Overall Accuracy : {overall_acc:.4f}")
print(f"\n── Full Classification Report ───────────────────────────────────────")
print(classification_report(all_true_cls, all_pred_cls, labels=[0, 1, 2, 3], zero_division=0,
                             target_names=class_names, digits=3))

from sklearn.metrics import f1_score

macro_f1    = f1_score(all_true_cls, all_pred_cls, average='macro',  zero_division=0)
weighted_f1 = f1_score(all_true_cls, all_pred_cls, average='weighted', zero_division=0)

print(f"  Macro F1         : {macro_f1:.4f}  "
      f"({'good' if macro_f1 > 0.5 else 'needs improvement'})")
print(f"  Weighted F1      : {weighted_f1:.4f}")

In [ ]:
# Make sure eval cell has already run so all_true_cls etc. are defined

common = dict(
    model=model_eval, dataset=te_ds, normalizer=normalizer_eval,
    n_samples=36,
    all_true_cls=all_true_cls, all_pred_cls=all_pred_cls,
    all_true_norm=all_true_norm, all_pred_norm=all_pred_norm,
)

# 1) Random mix (original behaviour)
plot_forecast_grid(**common, mode='random',       title='Random Samples')

# # 2) Missed flares — true M/X, model said Quiet/C
# plot_forecast_grid(**common, mode='missed',       title='Missed Flares')

# # 3) False alarms — true Quiet, model panicked
# plot_forecast_grid(**common, mode='false_alarm',  title='False Alarms')

# # 4) Worst peak-error predictions
# plot_forecast_grid(**common, mode='worst',        title='Worst Predictions')

# # 5) Best predictions on actual flare events
# plot_forecast_grid(**common, mode='best',         title='Best Flare Predictions')

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

assert 'model_eval' in dir(), "model_eval not found — run load_checkpoint_v4() first"
assert 'normalizer' in dir(), "normalizer not found"
assert 'df_feat'    in dir(), "df_feat not found — run data-loading cells first"

H     = HISTORY_STEPS
FCAST = FORECAST_STEPS
CS    = CADENCE_S

sx_log_raw = df_feat['sx_log'].values.astype(np.float32)
sx_phys    = np.expm1(sx_log_raw).clip(0.3, None)

X_raw  = df_feat[FEATURE_COLS_V4].values.astype(np.float32)
X_norm = normalizer.transform(X_raw)
N      = len(sx_phys)

t_min = np.arange(N) * CS / 60.0

N_win = N - H - FCAST
i_tr  = int(N_win * 0.70) + H
i_vl  = int(N_win * 0.85) + H

print(f"  Total timesteps : {N:,}  ({N*CS/3600:.1f} hours)")
print(f"  Train ends      : {t_min[i_tr]:.1f} min")
print(f"  Val ends        : {t_min[i_vl]:.1f} min")
print(f"  Model window    : {H*CS//60} min in → {FCAST*CS//60} min out")

max_start = N - H - FCAST - 1

def run_inference(start_idx):
    if start_idx < 0 or (start_idx + H + FCAST) > N:
        return None, None
    window   = X_norm[start_idx : start_idx + H]
    x_tensor = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_norm, cls_logits = model_eval(x_tensor)
    pred_norm = pred_norm.squeeze(0).cpu().numpy()
    pred_phys = normalizer.inverse_target(pred_norm).clip(0.3, None)
    act_phys  = sx_phys[start_idx + H : start_idx + H + FCAST]
    return pred_phys, act_phys

def classify_peak(peak):
    if   peak >= X_THRESHOLD: return 'X-Class',  '#8B0000'
    elif peak >= M_THRESHOLD: return 'M-Class',  '#FF4500'
    elif peak >= C_THRESHOLD: return 'C-Class',  '#FFA500'
    else:                     return 'Quiet',     '#228B22'

out = widgets.Output()

def update_plot(idx):
    idx = int(np.clip(idx, H, max_start))

    t_hist   = t_min[idx       : idx + H       ]
    t_future = t_min[idx + H   : idx + H + FCAST]
    hist_w   = sx_phys[idx     : idx + H       ]

    pred_phys, act_phys = run_inference(idx)
    if pred_phys is None:
        return

    pred_cls, pred_col = classify_peak(float(pred_phys.max()))
    act_cls,  act_col  = classify_peak(float(act_phys.max()))
    correct_sym = '✅' if pred_cls == act_cls else '❌'

    region = '🔵 Train' if idx < i_tr else ('🟠 Val' if idx < i_vl else '🟢 Test')

    with out:
        clear_output(wait=True)
        fig = plt.figure(figsize=(16, 8), facecolor='#fafbff')
        gs  = GridSpec(2, 1, figure=fig, height_ratios=[0.6, 0.4], hspace=0.38)

        ax1 = fig.add_subplot(gs[0])
        ax1.set_facecolor('#f8f9ff')

        # Region fills
        ax1.axvspan(t_min[0],    t_min[i_tr], alpha=0.08, color='royalblue', label='Train')
        ax1.axvspan(t_min[i_tr], t_min[i_vl], alpha=0.08, color='orange',    label='Val')
        ax1.axvspan(t_min[i_vl], t_min[-1],   alpha=0.08, color='limegreen', label='Test')

        # Full flux (downsampled for speed)
        step = max(1, N // 8000)
        ax1.semilogy(t_min[::step], sx_phys[::step],
                     color='royalblue', lw=0.6, alpha=0.8, label='SoLEXS flux')

        # Threshold lines
        ax1.axhline(C_THRESHOLD, color='orange',  ls='--', lw=1.2, label=f'C ({C_THRESHOLD:.0f})')
        ax1.axhline(M_THRESHOLD, color='purple',  ls='--', lw=1.2, label=f'M ({M_THRESHOLD:.0f})')

        # Selected window highlight
        ax1.axvspan(t_hist[0], t_future[-1], alpha=0.18, color='red',  label='Selected window')
        ax1.axvline(t_future[0], color='gray', ls=':', lw=1.5)

        ax1.set_xlim(t_min[0], t_min[-1])
        ax1.set_ylim(0.3, 600)
        ax1.set_ylabel('SoLEXS (counts/s)', fontsize=10)
        ax1.set_xlabel('Time (minutes)', fontsize=10)
        ax1.set_title('📈 Full SoLEXS Time Series', fontsize=11, fontweight='bold')
        ax1.legend(loc='upper right', fontsize=7, ncol=4,
                   framealpha=0.85, edgecolor='lightgray')
        ax1.grid(True, which='both', alpha=0.2)

        ax2 = fig.add_subplot(gs[1])
        ax2.set_facecolor('#f8f9ff')

        ax2.semilogy(t_hist,   hist_w,    color='royalblue', lw=2.0, label='History (60 min)')
        ax2.semilogy(t_future, act_phys,  color='#00AA44',   lw=2.5, label='Actual (10 min)')
        ax2.semilogy(t_future, pred_phys, color='red',       lw=2.5, ls='--', label='Predicted (10 min)')

        ax2.axhline(C_THRESHOLD, color='orange', ls='--', lw=1.0, alpha=0.7)
        ax2.axhline(M_THRESHOLD, color='purple', ls='--', lw=1.0, alpha=0.7)
        ax2.axvline(t_future[0], color='gray', ls=':', lw=1.5)

        all_y = np.concatenate([hist_w, act_phys, pred_phys])
        ax2.set_ylim(max(0.3, all_y.min() * 0.5), all_y.max() * 2.5)
        ax2.set_xlim(t_hist[0] - 2, t_future[-1] + 2)
        ax2.set_ylabel('SoLEXS (counts/s)', fontsize=10)
        ax2.set_xlabel('Time (minutes)', fontsize=10)
        ax2.set_title(
            f'🔬 Zoomed Window  |  {region}  |  '
            f'Input: {t_hist[0]:.1f}–{t_hist[-1]:.1f} min  →  '
            f'Forecast: {t_future[0]:.1f}–{t_future[-1]:.1f} min',
            fontsize=10, fontweight='bold'
        )

        # Prediction summary box
        summary = (
            f'Predicted: {pred_cls}  |  Actual: {act_cls}  {correct_sym}\n'
            f'Peak pred: {pred_phys.max():.1f} cts/s  |  Peak actual: {act_phys.max():.1f} cts/s'
        )
        ax2.text(0.01, 0.97, summary, transform=ax2.transAxes,
                 fontsize=9, va='top', ha='left',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                           edgecolor='#ccccdd', alpha=0.9))

        ax2.legend(loc='upper right', fontsize=8, framealpha=0.85, edgecolor='lightgray')
        ax2.grid(True, which='both', alpha=0.2)

        plt.tight_layout()
        plt.show()

position_slider = widgets.IntSlider(
    value=i_vl + 100,
    min=H,
    max=max_start,
    step=1,
    description='Window start:',
    continuous_update=False,
    style={'description_width': '115px'},
    layout=widgets.Layout(width='85%')
)

step_dropdown = widgets.Dropdown(
    options=[
        ('1 step  (10 s)',    1),
        ('6 steps (1 min)',   6),
        ('30 steps (5 min)',  30),
        ('60 steps (10 min)', 60),
        ('180 steps (30 min)', 180),
    ],
    value=6,
    description='Slider step:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='240px'),
)

prev_btn   = widgets.Button(description='◀ Prev',        layout=widgets.Layout(width='85px'))
next_btn   = widgets.Button(description='Next ▶',        layout=widgets.Layout(width='85px'))
jump_c_btn = widgets.Button(description='Jump → next C', button_style='warning',
                            layout=widgets.Layout(width='135px'))
jump_m_btn = widgets.Button(description='Jump → next M', button_style='danger',
                            layout=widgets.Layout(width='135px'))

def on_slider_change(change):
    update_plot(change['new'])

def on_prev(_):
    position_slider.value = max(H, position_slider.value - step_dropdown.value)

def on_next(_):
    position_slider.value = min(max_start, position_slider.value + step_dropdown.value)

def jump_to_next_event(min_threshold):
    start = position_slider.value + 1
    for i in range(start, max_start, 6):
        if sx_phys[i + H : i + H + FCAST].max() >= min_threshold:
            position_slider.value = i
            return
    print(f"No more events above {min_threshold:.0f} cts/s after current position.")

def on_jump_c(_): jump_to_next_event(C_THRESHOLD)
def on_jump_m(_): jump_to_next_event(M_THRESHOLD)

position_slider.observe(on_slider_change, names='value')
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
jump_c_btn.on_click(on_jump_c)
jump_m_btn.on_click(on_jump_m)

controls = widgets.VBox([
    widgets.HTML('<b style="font-size:13px;color:#1a1a2e;">📍 Window Control</b>'),
    position_slider,
    widgets.HBox([
        prev_btn, next_btn,
        widgets.HTML('&nbsp;&nbsp;'),
        step_dropdown,
        widgets.HTML('&nbsp;&nbsp;'),
        jump_c_btn, jump_m_btn,
    ], layout=widgets.Layout(align_items='center')),
], layout=widgets.Layout(
    padding='10px', border='1px solid #dde',
    margin='4px 0px',
))

display(widgets.VBox([controls, out]))
update_plot(position_slider.value)
print("✓ Explorer ready — use slider or buttons to navigate.")
